# 🌦️ เปรียบเทียบพยากรณ์อากาศจากหลายโมเดล (Multi-Model Weather Comparison)

ดึงค่าพยากรณ์จาก **โมเดลระดับโลก 11 โมเดล** และ **ระบบ ensemble 6 ระบบ** ผ่าน [Open-Meteo API](https://open-meteo.com/) (ฟรี ไม่ต้องใช้ API key)
และ **TMD WRF 2 กม.** ของกรมอุตุนิยมวิทยา (ต้องใช้ token ฟรี) โดยเน้นโมเดลที่ **ความละเอียดเชิงพื้นที่สูง**
แล้วเปรียบเทียบกันที่ **ตำแหน่งของคุณ** และ **ตำแหน่งที่ต้องการ** จากนั้น **บันทึกว่าฝนตกจริงหรือไม่** เพื่อวิเคราะห์ย้อนกลับว่าโมเดลไหนแม่น

**พารามิเตอร์:** ฝน 🌧️ · โอกาสฝน · อุณหภูมิสูง/ต่ำสุด 🌡️ · อุณหภูมิที่รู้สึก (Heat Index) 🥵 · ความชื้น 💧 · ลม/ลมกระโชก 💨 · เมฆ ☁️ · UV ☀️ · PM2.5 😷

| ช่วงเวลาล่วงหน้า | ฝน | พารามิเตอร์อื่น |
|---|---|---|
| **+3, +6, +12, +24 ชม.** | ฝน **สะสม** ตั้งแต่ตอนนี้ถึง +N ชม. | ค่า **ณ เวลา** +N ชม. (ลมกระโชก/โอกาสฝนใช้ค่าสูงสุดในช่วง) |
| **D+1, D+3, D+7** | ฝน **รวมทั้งวัน** | ค่าสูงสุด/ต่ำสุด/เฉลี่ย **ทั้งวัน** |

| ไฟล์ | หน้าที่ |
|---|---|
| `config.json` | ตำแหน่ง, threshold, รอบการอัปเดต |
| `weather_core.py` | ดึงข้อมูลและสรุปผล (ใช้ซ้ำในแอปได้) |
| `weather_store.py` | ฐานข้อมูล SQLite, บันทึกฝนจริง, คำนวณคะแนนความแม่นยำ |
| `collector.py` | ดึงอัตโนมัติจาก command line / launchd / cron |
| 📖 **[docs/MODELS.md](docs/MODELS.md)** | แหล่งข้อมูล หลักการของแต่ละโมเดล และวิธีอ่านค่า |
| 📖 **[README.md](README.md)** | วิธีติดตั้งและใช้งาน |

**สารบัญ:** ① ตั้งค่า → ② โมเดล → ③ พิกัด → ④ ฟังก์ชันแสดงผล → ⑤ เปรียบเทียบทุกตำแหน่ง → ⑥ แผนที่ → ⑦ สรุปรวม → ⑧ อัปเดตอัตโนมัติ → ⑨ บันทึกฝนจริง → ⑩ วิเคราะห์ความแม่นยำ

In [ ]:
# %pip install requests pandas matplotlib jinja2 folium branca ipywidgets   # ถ้ายังไม่มี ให้เอา # ออกแล้วรัน

import re
import threading

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager
from IPython.display import display, Markdown, clear_output
import folium
import branca.colormap as bcm
import ipywidgets as W

import weather_core as core
import weather_store as store

# ฟอนต์ไทยสำหรับกราฟ (macOS: Sukhumvit Set, Windows: Leelawadee UI/Tahoma, Linux: Noto Sans Thai)
# หมายเหตุ: Thonburi บน macOS วางสระ/วรรณยุกต์เพี้ยนใน matplotlib จึงไม่ใช้
_installed = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams["font.family"] = [f for f in ["Sukhumvit Set", "Leelawadee UI", "Noto Sans Thai", "Tahoma"]
                               if f in _installed][:1] + ["DejaVu Sans"]

pd.set_option("display.precision", 1)
pd.set_option("display.max_columns", 30)

## ① ตั้งค่า

ค่าเริ่มต้นอ่านจาก `config.json` ถ้าแก้ในเซลล์นี้จะมีผลเฉพาะ notebook ที่เปิดอยู่ (collector ยังอ่านจาก `config.json`)

- `my_location = "auto"` หาตำแหน่งจาก IP address (แม่นระดับเมือง) ถ้าต้องการแม่นกว่านี้ให้ใส่ `[lat, lon, "ชื่อ"]`
- `target_places` ใส่ได้ทั้งชื่อสถานที่ (ไทย/อังกฤษ) และ `[lat, lon, "ชื่อ"]`
- `rain_threshold_mm` คือปริมาณฝนที่นับว่า "ฝนตก" ในตารางเปรียบเทียบและการคิดความน่าจะเป็นจาก ensemble
- `verify_window_hours` / `verify_threshold_mm` ใช้ตัดสินว่าโมเดล "ทายว่าฝนตก" ตอนวิเคราะห์ความแม่นยำ (ดูข้อ ⑩)
- `max_grid_km` ตัดโมเดลที่กริดหยาบกว่านี้ออก เช่น `16` จะเหลือแค่โมเดล 9–16 กม. (ค่า `None` = ใช้ทุกโมเดล)
- `tmd_token` ใส่ token ของ [TMD NWP API](https://data.tmd.go.th/nwpapi/register) (สมัครฟรี) เพื่อเพิ่ม **TMD WRF 2 กม.** หรือตั้ง environment variable `TMD_NWP_TOKEN` แทน (ปลอดภัยกว่า)

In [ ]:
CFG = core.CFG
# CFG["target_places"] = ["เชียงใหม่", "ภูเก็ต", "หาดใหญ่"]   # ตัวอย่างการแก้เฉพาะใน notebook
# CFG["my_location"] = [13.7563, 100.5018, "บ้าน"]
# CFG["max_grid_km"] = 16                                      # ใช้เฉพาะโมเดลความละเอียดสูง
# CFG["tmd_token"] = "..."                                     # หรือ export TMD_NWP_TOKEN=... ก่อนเปิด notebook

# แผนที่กริด (ครอบคลุมประเทศไทย)
MAP_BBOX = (5.5, 20.5, 97.5, 105.5)          # lat_min, lat_max, lon_min, lon_max
MAP_STEP = 1.0                               # องศา ถ้าลดลงจะละเอียดขึ้นแต่ใช้โควตา API มากขึ้น
MAP_DAY = 1                                  # แสดงวัน D+N
MAP_VARIABLE = "precipitation_sum"           # หรือ "temperature_2m_max", "apparent_temperature_max"

# แผนที่ละเอียดรอบตำแหน่งของฉัน (ใช้เฉพาะโมเดลกริด ≤ LOCAL_MAX_GRID_KM)
LOCAL_MAP = True
LOCAL_RADIUS_DEG = 0.4                       # ±0.4° ≈ ±45 กม.
LOCAL_STEP = 0.1                             # 0.1° ≈ 11 กม. (81 จุด)
LOCAL_MAX_GRID_KM = 16

CHART_DAYS = 7
OUTPUT_DIR = core.ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
{k: v for k, v in CFG.items()}

## ①-ข เลือกตำแหน่ง (แผนที่ / พิกัด / ลิงก์ Google Maps)

- **คลิกบนแผนที่** หรือ **ลากหมุด 📍** ไปจุดที่ต้องการ
- หรือ **วาง** `13.7563, 100.5018` / ลิงก์ Google Maps / ชื่อสถานที่ ในช่อง "วาง" แล้วกด Enter
  (ใน Google Maps ให้คลิกขวาที่จุด แล้วกดพิกัดบรรทัดแรกเพื่อคัดลอก)
- ตั้งชื่อ แล้วกด **ตั้งเป็นตำแหน่งของฉัน** หรือ **เพิ่มตำแหน่งที่ต้องการ** จากนั้นกด **💾 บันทึก**
- ถ้าใช้ GitHub Actions ให้กด **⬆️ push ขึ้น GitHub** ด้วย รอบถัดไปจะดึงตำแหน่งใหม่เอง
  (repo เป็น public พิกัดที่บันทึกจะเปิดให้คนอื่นเห็นได้)
- แก้ผ่านหน้าเว็บ GitHub ก็ได้ ที่แท็บ **Actions → ตั้งค่าตำแหน่ง → Run workflow**

In [ ]:
import subprocess

_lbl = lambda x: x if isinstance(x, str) else x[2]
PICK = {"me": CFG["my_location"], "targets": list(CFG["target_places"])}
_geo_cache = {}


def _coords_of(loc):
    # คืน (lat, lon) ของตำแหน่งใน config (ชื่อสถานที่ต้อง geocode ก่อน)
    if loc == "auto":
        return None
    if isinstance(loc, str):
        if loc not in _geo_cache:
            try:
                _geo_cache[loc] = core.geocode(loc)[:2]
            except Exception:
                _geo_cache[loc] = None
        return _geo_cache[loc]
    return tuple(loc[:2])


p_paste = W.Text(description="วาง", placeholder="13.7563, 100.5018 หรือลิงก์ Google Maps หรือชื่อสถานที่ แล้วกด Enter",
                 layout={"width": "620px"})
p_lat = W.FloatText(description="lat", layout={"width": "200px"})
p_lon = W.FloatText(description="lon", layout={"width": "200px"})
p_name = W.Text(description="ชื่อ", placeholder="บ้าน / ที่ทำงาน", layout={"width": "260px"})
b_me = W.Button(description="🏠 ตั้งเป็นตำแหน่งของฉัน", layout={"width": "210px"})
b_add = W.Button(description="➕ เพิ่มตำแหน่งที่ต้องการ", layout={"width": "210px"})
p_remove = W.Dropdown(description="ลบ", layout={"width": "320px"})
b_remove = W.Button(description="🗑️ ลบ", layout={"width": "80px"})
b_save = W.Button(description="💾 บันทึกลง config.json", button_style="primary", layout={"width": "210px"})
b_push = W.Button(description="⬆️ push ขึ้น GitHub", layout={"width": "180px"})
p_msg = W.HTML()
p_list = W.HTML()
_busy = {"sync": False}


def _set_point(lat, lon, source=None):
    # อัปเดตช่อง lat/lon และหมุดให้ตรงกัน
    if _busy["sync"]:
        return
    _busy["sync"] = True
    try:
        p_lat.value, p_lon.value = round(lat, 5), round(lon, 5)
        if source != "marker" and _MAP is not None:
            _pick_marker.location = (lat, lon)
        warn = "" if core.in_tmd_domain(lat, lon) else " ⚠️ นอกพื้นที่ TMD WRF (มีเฉพาะโมเดลระดับโลก)"
        p_msg.value = f"📍 เลือก {lat:.5f}, {lon:.5f}{warn}"
    finally:
        _busy["sync"] = False


def _refresh_list():
    me = PICK["me"]
    me_txt = "auto (จาก IP)" if me == "auto" else (me if isinstance(me, str) else f"{me[2]} ({me[0]:.4f}, {me[1]:.4f})")
    rows = "".join(f"<li>{_lbl(t)}" + ("" if isinstance(t, str) else f" ({t[0]:.4f}, {t[1]:.4f})") + "</li>"
                   for t in PICK["targets"])
    p_list.value = f"<b>🏠 ตำแหน่งของฉัน:</b> {me_txt}<br><b>📌 ตำแหน่งที่ต้องการ:</b><ul style='margin:0'>{rows}</ul>"
    p_remove.options = [_lbl(t) for t in PICK["targets"]]
    if _MAP is not None:
        _saved.clear_layers()
        for loc, color in [(PICK["me"], "red")] + [(t, "blue") for t in PICK["targets"]]:
            xy = _coords_of(loc)
            if xy:
                _saved.add_layer(CircleMarker(location=xy, radius=7, color=color, fill_color=color,
                                              fill_opacity=0.8, title=_lbl(loc) if loc != "auto" else "auto"))


def _current():
    name = p_name.value.strip() or f"{p_lat.value:.3f},{p_lon.value:.3f}"
    return [round(p_lat.value, 5), round(p_lon.value, 5), name]


def _on_paste(_):
    xy = core.parse_coords(p_paste.value)
    try:
        if xy is None:
            la, lo, found = core.geocode(p_paste.value.strip())
            xy = (la, lo)
            p_name.value = p_name.value or found
        _set_point(*xy)
        if _MAP is not None:
            _MAP.center = xy
            _MAP.zoom = max(_MAP.zoom, 11)
    except Exception as e:
        p_msg.value = f"⚠️ อ่านพิกัดไม่ได้: {e}"


def _set_me(_):
    PICK["me"] = _current()
    p_msg.value = f"🏠 ตั้งตำแหน่งของฉัน = {PICK['me'][2]} (ยังไม่บันทึก)"
    _refresh_list()


def _add(_):
    loc = _current()
    PICK["targets"] = [t for t in PICK["targets"] if _lbl(t) != loc[2]] + [loc]
    p_msg.value = f"➕ เพิ่ม {loc[2]} (ยังไม่บันทึก)"
    _refresh_list()


def _remove(_):
    PICK["targets"] = [t for t in PICK["targets"] if _lbl(t) != p_remove.value]
    p_msg.value = f"🗑️ ลบ {p_remove.value} (ยังไม่บันทึก)"
    _refresh_list()


def _save(_):
    core.save_locations(PICK["me"], PICK["targets"])
    p_msg.value = "✅ บันทึกลง config.json แล้ว · รันเซลล์ ③ ใหม่เพื่อใช้ตำแหน่งใหม่ · กด push ถ้าใช้ GitHub Actions"


def _push(_):
    _save(None)
    cmds = [["git", "add", "config.json"],
            ["git", "commit", "-m", "location: update from notebook"],
            ["git", "pull", "--rebase", "-q"],
            ["git", "push", "-q"]]
    for c in cmds:
        r = subprocess.run(c, cwd=core.ROOT, capture_output=True, text=True)
        if r.returncode != 0 and not (c[1] == "commit" and "nothing to commit" in r.stdout):
            p_msg.value = f"⚠️ {' '.join(c)}: {(r.stderr or r.stdout).strip()[:300]}"
            return
    p_msg.value = "✅ push ขึ้น GitHub แล้ว รอบถัดไปของ GitHub Actions จะใช้ตำแหน่งใหม่"


p_paste.on_submit(_on_paste)
b_me.on_click(_set_me)
b_add.on_click(_add)
b_remove.on_click(_remove)
b_save.on_click(_save)
b_push.on_click(_push)
p_lat.observe(lambda ch: _set_point(p_lat.value, p_lon.value), names="value")
p_lon.observe(lambda ch: _set_point(p_lat.value, p_lon.value), names="value")

try:
    from ipyleaflet import Map, Marker, CircleMarker, LayerGroup, ScaleControl
    _start = _coords_of(PICK["me"]) or (13.75, 100.5)
    _MAP = Map(center=_start, zoom=6, scroll_wheel_zoom=True, layout={"height": "450px"})
    _MAP.add(ScaleControl(position="bottomleft"))
    _saved = LayerGroup()
    _MAP.add(_saved)
    _pick_marker = Marker(location=_start, draggable=True, title="ตำแหน่งที่เลือก")
    _MAP.add(_pick_marker)
    _MAP.on_interaction(lambda **kw: _set_point(*kw["coordinates"]) if kw.get("type") == "click" else None)
    _pick_marker.observe(lambda ch: _set_point(*ch["new"], source="marker"), names="location")
    _map_widget = _MAP
except ImportError:
    # ไม่มี ipyleaflet: ใช้แผนที่ folium ที่คลิกแล้วแสดงพิกัด ให้คัดลอกมาวางเอง
    _MAP = None
    _fm = folium.Map(location=(13.75, 100.5), zoom_start=6)
    _fm.add_child(folium.LatLngPopup())
    _map_widget = W.HTML(_fm._repr_html_())
    p_msg.value = "ℹ️ ติดตั้ง ipyleaflet เพื่อคลิกเลือกได้โดยตรง (ตอนนี้ให้คลิกแผนที่ แล้วคัดลอกพิกัดมาวาง)"

_refresh_list()
display(W.VBox([
    _map_widget,
    p_paste,
    W.HBox([p_lat, p_lon, p_name]),
    W.HBox([b_me, b_add]),
    W.HBox([p_remove, b_remove]),
    W.HBox([b_save, b_push]),
    p_msg, p_list,
]))

## ② โมเดล ความละเอียด และตัวแปร

`grid_km` คือระยะห่างกริดที่ **วัดจริง** ที่ประเทศไทย (ดู docs/MODELS.md § 3) ตารางเรียงจากละเอียดไปหยาบ
ถ้ามี token ของ TMD ตัว **TMD WRF (2 กม.)** จะเป็นคอลัมน์แรกของทุกตาราง

ระยะพยากรณ์จริง **ไม่ได้ hard-code** แต่ตรวจจากข้อมูลที่ได้มา ถ้าโมเดลไหนพยากรณ์ไปไม่ถึงช่วงเวลานั้นหรือไม่มีตัวแปรนั้น ช่องจะแสดง `–`
โมเดลที่ไม่มีข้อมูลเลยในรอบนั้น (เช่น KMA, BOM ในบางช่วง) จะถูกตัดออกอัตโนมัติ

In [ ]:
MODELS = core.model_table(CFG)
display(MODELS[["open_meteo_key", "agency", "type", "grid", "grid_km", "ใช้งาน"]])
display(pd.DataFrame(core.ENSEMBLE_MODELS).T.rename_axis("open_meteo_key").sort_values("grid_km"))

fig, ax = plt.subplots(figsize=(9, 4.5))
colors = ["#2ca02c" if u else "#bbbbbb" for u in MODELS["ใช้งาน"]]
ax.barh(MODELS.index, MODELS["grid_km"], color=colors)
for i, v in enumerate(MODELS["grid_km"]):
    ax.text(v + 0.5, i, f"{v} กม.", va="center", fontsize=9)
ax.axvline(4, color="red", ls="--", lw=1, label="≤ 4 กม. = จำลองพายุฝนฟ้าคะนองได้โดยตรง")
ax.legend(loc="lower right", fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("ระยะห่างกริด (กม.) — ยิ่งน้อยยิ่งละเอียด")
ax.set_title("ความละเอียดเชิงพื้นที่ของแต่ละโมเดล (สีเทา = ยังไม่ได้ใช้)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()
display(pd.DataFrame(core.VARIABLES).T[["api", "label", "unit", "hour", "day"]])

## ③ หาพิกัดตำแหน่ง

In [ ]:
LOCATIONS = core.resolve_locations(CFG)
pd.DataFrame(LOCATIONS, columns=["lat", "lon", "name"])

## ④ ฟังก์ชันแสดงผล (ตาราง + กราฟ)

In [ ]:
def style_tbl(df, cmap="Blues", fmt="{:.1f}"):
    try:
        return df.style.format(fmt, na_rep="–").background_gradient(cmap=cmap, axis=None)
    except ImportError:          # ไม่มี jinja2
        return df


def plot_heatmap(table, title, unit="มม.", cmap="Blues", vmax=None, ax=None, fmt=None, counts=None):
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(1.3 * table.shape[1] + 3, 0.45 * table.shape[0] + 1.5))
    vals = table.values.astype(float)
    im = ax.imshow(np.ma.masked_invalid(vals), cmap=cmap, aspect="auto", vmin=0, vmax=vmax)
    ax.set_xticks(range(table.shape[1]), [str(c) for c in table.columns], fontsize=9)
    ax.set_yticks(range(table.shape[0]), table.index, fontsize=9)
    fmt = fmt or ("{:.0f}" if unit == "%" else "{:.1f}")
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            v = vals[i, j]
            txt = "–" if np.isnan(v) else fmt.format(v)
            if counts is not None and not np.isnan(v):
                txt += f"\n(n={int(counts.values[i, j])})"
            rgb = im.cmap(im.norm(v))[:3] if not np.isnan(v) else (1, 1, 1)
            dark = 0.299 * rgb[0] + 0.587 * rgb[1] + 0.114 * rgb[2] < 0.5      # เลือกสีตัวอักษรตามความสว่างของช่อง
            ax.text(j, i, txt, ha="center", va="center", fontsize=8, color="white" if dark else "black")
    ax.set_title(title, fontsize=11)
    plt.colorbar(im, ax=ax, label=unit)
    if own:
        plt.tight_layout()
        plt.show()


def plot_meteogram(r, days=CHART_DAYS):
    # กราฟรายชั่วโมงหลายโมเดล: ฝน, อุณหภูมิ, รู้สึกเหมือน, ความชื้น, ลมกระโชก, PM2.5
    data, aq, now = r["data"], r["aq"], r["now"]
    end = now + pd.Timedelta(days=days)
    panels = [("precipitation", "ฝน (มม./ชม.)"), ("temperature_2m", "อุณหภูมิ (°C)"),
              ("apparent_temperature", "รู้สึกเหมือน (°C)"), ("relative_humidity_2m", "ความชื้น (%)"),
              ("wind_gusts_10m", "ลมกระโชก (กม./ชม.)")]
    n = len(panels) + (1 if aq is not None else 0)
    fig, axes = plt.subplots(n, 1, figsize=(13, 2.3 * n), sharex=True)
    colors = dict(zip(core.MODEL_NAMES, plt.cm.tab20(np.linspace(0, 1, len(core.MODEL_NAMES)))))
    for ax, (var, ylabel) in zip(axes, panels):
        df = data.get(var, pd.DataFrame())
        df = df[(df.index > now) & (df.index <= end)]
        for col in df:
            if col == core.TMD_MODEL["name"]:          # โมเดลละเอียดสุด (2 กม.) เน้นให้เห็นชัด
                ax.plot(df.index, df[col], lw=2.2, color="crimson", label=f"{col} (2 กม.)", zorder=5)
            else:
                ax.plot(df.index, df[col], lw=1, alpha=0.75, color=colors.get(col), label=col)
        if not df.empty:
            ax.plot(df.index, df.median(axis=1), color="black", lw=2.2, label="Median")
        if var == "apparent_temperature":
            for lvl, c in [(33, "gold"), (42, "orange"), (52, "red")]:
                ax.axhline(lvl, color=c, ls="--", lw=1)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.grid(alpha=0.3)
    axes[0].legend(ncol=6, fontsize=7, loc="lower left", bbox_to_anchor=(0, 1.02))
    if aq is not None:
        a = aq[(aq.index > now) & (aq.index <= end)]
        axes[-1].fill_between(a.index, a["pm2_5"], color="grey", alpha=0.5)
        axes[-1].axhline(37.5, color="orange", ls="--", lw=1)
        axes[-1].set_ylabel("PM2.5 (CAMS)", fontsize=9)
        axes[-1].grid(alpha=0.3)
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d/%m\n%H:%M"))
    fig.suptitle(f"{r['name']}: พยากรณ์รายชั่วโมง {days} วัน แยกโมเดล")
    plt.tight_layout()
    plt.show()


def plot_daily_bars(r):
    # กราฟแท่ง: ฝนรายวัน, อุณหภูมิสูงสุด, รู้สึกเหมือน ของ D+1/3/7 แยกโมเดล
    tables = r["tables"]
    day_cols = [c for c in tables["rain"].columns if c.startswith("D+")]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, key in zip(axes, ["rain", "tmax", "feels"]):
        t = tables.get(key)
        if t is None or t.empty:
            continue
        t[[c for c in day_cols if c in t]].T.plot.bar(ax=ax, width=0.85, legend=False, colormap="tab20")
        ax.set_title(f"{core.VARIABLES[key]['label']} ({core.VARIABLES[key]['unit']})", fontsize=10)
        ax.tick_params(axis="x", rotation=0)
        ax.grid(axis="y", alpha=0.3)
        if key != "rain":
            ax.set_ylim(np.nanmin(t.values) - 3, np.nanmax(t.values) + 2)
    axes[0].legend(fontsize=7, ncol=2)
    fig.suptitle(f"{r['name']}: เปรียบเทียบรายวันแยกโมเดล")
    plt.tight_layout()
    plt.show()


def render_location(r, charts=True):
    tables = r["tables"]
    thr = CFG["rain_threshold_mm"]
    display(Markdown(f"---\n## 📍 {r['name']}  ({r['lat']:.3f}, {r['lon']:.3f})\n"
                     f"เวลาอ้างอิง: **{r['now']:%Y-%m-%d %H:%M}** ({r['tz']}) · "
                     f"โมเดลที่มีข้อมูลฝน: **{r['data']['precipitation'].shape[1]}** "
                     f"(ละเอียดสุด: {r['data']['precipitation'].columns[0]})"))
    display(Markdown("### 🗓️ สรุปรายวัน (median ของทุกโมเดล)"))
    display(r["brief"])
    display(Markdown("### 🌧️ ปริมาณฝนพยากรณ์ (มม.) แยกโมเดล"))
    display(style_tbl(tables["rain"]))
    display(Markdown("**ฉันทามติระหว่างโมเดล**"))
    display(r["consensus"])
    display(Markdown(f"**โอกาสฝน ≥ {thr} มม. จาก ensemble (%)** (เลขในวงเล็บคือจำนวนสมาชิก)"))
    display(style_tbl(r["ensemble"], "Greens", "{:.0f}"))
    if not charts:
        return
    display(Markdown("### 🌡️ พารามิเตอร์อื่น ๆ แยกโมเดล"))
    for k, t in tables.items():
        if k == "rain":
            continue
        v = core.VARIABLES[k]
        display(Markdown(f"**{v['label']}** ({v['unit']})"))
        display(style_tbl(t, v["cmap"], "{:.0f}"))
    fig, axes = plt.subplots(1, 2, figsize=(17, 0.45 * len(tables["rain"]) + 2))
    plot_heatmap(tables["rain"], "ฝนพยากรณ์แยกโมเดล (มม.)", ax=axes[0])
    if not r["ensemble"].empty:
        plot_heatmap(r["ensemble"], f"โอกาสฝน ≥ {thr} มม. (ensemble)", unit="%", cmap="Greens", vmax=100, ax=axes[1])
    fig.suptitle(r["name"])
    plt.tight_layout()
    plt.show()
    plot_daily_bars(r)
    plot_meteogram(r)


def save_csv(r):
    safe = re.sub(r'[\\/:*?"<>|(),\s]+', "_", r["name"]).strip("_")
    stamp = f"{r['now']:%Y%m%d_%H%M}"
    for k, t in r["tables"].items():
        t.to_csv(OUTPUT_DIR / f"{safe}_{stamp}_{k}.csv", encoding="utf-8-sig")
    r["ensemble"].to_csv(OUTPUT_DIR / f"{safe}_{stamp}_ensemble.csv", encoding="utf-8-sig")
    r["brief"].to_csv(OUTPUT_DIR / f"{safe}_{stamp}_brief.csv", encoding="utf-8-sig")

## ⑤ เปรียบเทียบทุกตำแหน่ง

ทุกครั้งที่รันเซลล์นี้ ผลพยากรณ์จะถูก **บันทึกลงฐานข้อมูล** (`data/weather.db`) ด้วย เพื่อใช้วิเคราะห์ความแม่นยำย้อนหลังในข้อ ⑩

In [ ]:
RESULTS = {}
for lat, lon, name in LOCATIONS:
    r = core.analyze_location(lat, lon, name, CFG)
    RESULTS[name] = r
    store.save_snapshot(r, CFG)
    save_csv(r)
    render_location(r)

## ⑥ แผนที่ (Map)

แผนที่ interactive (folium) มี 2 ส่วน
1. **หมุดของแต่ละตำแหน่ง** คลิกแล้วเห็นสรุปรายวันกับตารางฝนแยกโมเดล สีหมุดบอกปริมาณฝน median ของวัน D+`MAP_DAY`
2. **กริดทั่วประเทศไทย** แยกเป็นเลเยอร์ละ 1 โมเดล และมีเลเยอร์ Median สลับดูได้จากปุ่มเลเยอร์มุมขวาบน

แผนที่จะบันทึกเป็นไฟล์ `outputs/map.html` ด้วย เปิดในเบราว์เซอร์ได้เลย

In [ ]:
MAP_UNITS = {"precipitation_sum": ("ฝนรวม", "มม."), "temperature_2m_max": ("อุณหภูมิสูงสุด", "°C"),
             "apparent_temperature_max": ("รู้สึกเหมือน (สูงสุด)", "°C")}


def build_map(results, grid=None, variable=MAP_VARIABLE, day=MAP_DAY, step=MAP_STEP, zoom=6):
    center = [np.mean([r["lat"] for r in results.values()]), np.mean([r["lon"] for r in results.values()])]
    fmap = folium.Map(location=center, zoom_start=zoom, tiles="CartoDB positron")
    label, unit = MAP_UNITS.get(variable, (variable, ""))

    if grid is not None:
        cols = [c for c in grid.columns if c not in ("lat", "lon")]
        if variable == "precipitation_sum":
            cmap = bcm.LinearColormap(["#ffffff", "#c6dbef", "#6baed6", "#2171b5", "#08306b", "#7a0177"],
                                      index=[0, 1, 5, 15, 35, 70], vmin=0, vmax=70)
        else:
            vals = grid[cols].values.astype(float)
            cmap = bcm.linear.YlOrRd_09.scale(np.nanpercentile(vals, 2), np.nanpercentile(vals, 98))
        cmap.caption = f"{label} D+{day} ({unit})"
        for i, col in enumerate(["Median"] + [c for c in cols if c != "Median"]):
            fg = folium.FeatureGroup(name=f"กริด: {col}", show=(i == 0))
            for la, lo, v in grid[["lat", "lon", col]].itertuples(index=False, name=None):
                if pd.isna(v):
                    continue
                h = step / 2
                folium.Rectangle(
                    bounds=[[la - h, lo - h], [la + h, lo + h]],
                    fill=True, fill_color=cmap(min(v, cmap.vmax)), fill_opacity=0.6, weight=0,
                    tooltip=f"{col}: {v:.1f} {unit} ({la:.1f}, {lo:.1f})",
                ).add_to(fg)
            fg.add_to(fmap)
        cmap.add_to(fmap)

    rain_cmap = bcm.LinearColormap(["green", "gold", "orange", "red", "purple"], index=[0, 1, 10, 35, 70], vmin=0, vmax=70)
    pins = folium.FeatureGroup(name="ตำแหน่งที่เลือก", show=True)
    for name, r in results.items():
        rain_t = r["tables"]["rain"]
        dcol = [c for c in rain_t.columns if c.startswith(f"D+{day} ")]
        med = rain_t[dcol[0]].median() if dcol else np.nan
        html = (f"<h4 style='margin:0'>{name}</h4>"
                f"<b>สรุปรายวัน (median)</b>{r['brief'].to_html(na_rep='–', float_format=lambda v: f'{v:.1f}')}"
                f"<b>ฝนแยกโมเดล (มม.)</b>{rain_t.to_html(na_rep='–', float_format=lambda v: f'{v:.1f}')}")
        folium.CircleMarker(
            location=[r["lat"], r["lon"]], radius=12, weight=2, color="black",
            fill=True, fill_color=rain_cmap(0 if pd.isna(med) else min(med, 70)), fill_opacity=0.9,
            tooltip=f"{name}: ฝน D+{day} median {med:.1f} มม.",
            popup=folium.Popup(folium.IFrame(html, width=720, height=420), max_width=740),
        ).add_to(pins)
    pins.add_to(fmap)
    folium.LayerControl(collapsed=False).add_to(fmap)
    return fmap


try:
    GRID = core.fetch_grid(MAP_VARIABLE, MAP_DAY, step=MAP_STEP, bbox=MAP_BBOX)
except Exception as e:
    print(f"⚠️ ดึงกริดไม่ได้: {e}")
    GRID = None

MAP = build_map(RESULTS, GRID)
MAP.save(str(OUTPUT_DIR / "map.html"))
MAP

In [ ]:
# แผนที่กริดแบบภาพนิ่ง (small multiples) ดูทุกโมเดลพร้อมกันในรูปเดียว
if GRID is not None:
    cols = ["Median"] + [c for c in GRID.columns if c not in ("lat", "lon", "Median")]
    ncol = 5
    nrow = int(np.ceil(len(cols) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 4.6 * nrow), squeeze=False)
    label, unit = MAP_UNITS.get(MAP_VARIABLE, (MAP_VARIABLE, ""))
    is_rain = MAP_VARIABLE == "precipitation_sum"
    allv = GRID[cols].values.astype(float)
    vmin, vmax = (0, 50) if is_rain else (np.nanpercentile(allv, 2), np.nanpercentile(allv, 98))
    for ax, col in zip(axes.flat, cols):
        piv = GRID.pivot(index="lat", columns="lon", values=col)
        im = ax.pcolormesh(piv.columns, piv.index, piv.values, cmap="Blues" if is_rain else "YlOrRd",
                           vmin=vmin, vmax=vmax, shading="nearest")
        for r in RESULTS.values():
            ax.plot(r["lon"], r["lat"], "r^", ms=6)
        ax.set_title(col, fontsize=10)
        ax.set_aspect("equal")
        ax.tick_params(labelsize=7)
    for ax in axes.flat[len(cols):]:
        ax.axis("off")
    fig.colorbar(im, ax=axes, shrink=0.6, label=f"{label} ({unit})")
    fig.suptitle(f"{label} D+{MAP_DAY} แยกโมเดล (▲ = ตำแหน่งที่เลือก)")
    plt.show()

### แผนที่ละเอียดรอบตำแหน่งของฉัน

กริด 0.1° (≈11 กม.) รอบตำแหน่งแรกใน `LOCATIONS` ใช้เฉพาะโมเดลกริด ≤ `LOCAL_MAX_GRID_KM` กม. เพื่อดูว่าโมเดลละเอียดสูงเห็นฝน **ต่างกันตามพื้นที่** อย่างไร
> ถ้ากริดแผนที่ละเอียดกว่ากริดโมเดล (เช่น GFS 13 กม. บนแผนที่ 11 กม.) ช่องที่ติดกันจะได้ค่าเดียวกัน เพราะเป็นช่องเดียวกันของโมเดล

In [ ]:
if LOCAL_MAP:
    lat0, lon0, name0 = LOCATIONS[0]
    hi_res = [k for k, m in core.active_models(CFG).items()
              if m["grid_km"] <= LOCAL_MAX_GRID_KM and k not in ("kma_gdps", "bom_access_global")]
    bbox = (lat0 - LOCAL_RADIUS_DEG, lat0 + LOCAL_RADIUS_DEG, lon0 - LOCAL_RADIUS_DEG, lon0 + LOCAL_RADIUS_DEG)
    try:
        LGRID = core.fetch_grid(MAP_VARIABLE, MAP_DAY, models=hi_res, step=LOCAL_STEP, bbox=bbox)
        LMAP = build_map({name0: RESULTS[name0]}, LGRID, step=LOCAL_STEP, zoom=9)
        LMAP.save(str(OUTPUT_DIR / "map_local.html"))
        display(LMAP)
    except Exception as e:
        print(f"⚠️ ดึงกริดละเอียดไม่ได้: {e}")

## ⑦ ตารางสรุปรวมทุกตำแหน่ง

In [ ]:
overview = pd.DataFrame({n: r["consensus"].loc["median"] for n, r in RESULTS.items()}).T
agree = pd.DataFrame({n: r["consensus"].loc["% เห็นตรงกันว่าฝนตก"] for n, r in RESULTS.items()}).T
display(Markdown("**Median ฝนพยากรณ์ (มม.)**"))
display(style_tbl(overview))
display(Markdown(f"**% โมเดลที่เห็นตรงกันว่าฝน ≥ {CFG['rain_threshold_mm']} มม.**"))
display(style_tbl(agree, "Greens", "{:.0f}"))
display(Markdown("**สรุป D+1 ทุกตำแหน่ง**"))
display(pd.DataFrame({n: r["brief"].iloc[:, 0] for n, r in RESULTS.items()}))
overview.to_csv(OUTPUT_DIR / "overview_median.csv", encoding="utf-8-sig")

## ⑧ อัปเดตข้อมูลอัตโนมัติ

มี 2 แบบ

**แบบ A — ใน notebook (ทำงานเฉพาะตอนที่ notebook เปิดอยู่)**
เซลล์ด้านล่างรัน thread เบื้องหลังเพื่อดึงข้อมูลทุก `collect_every_minutes` นาที บันทึกลงฐานข้อมูล แล้วอัปเดตแดชบอร์ดสรุปในกล่องด้านล่างให้เอง

**แบบ B — เบื้องหลังของเครื่อง (แนะนำ ทำงานแม้ไม่ได้เปิด notebook)**
```bash
python3 collector.py install-launchd --every 60   # macOS: สร้างไฟล์ launchd แล้วทำตามคำสั่งที่แสดง
# หรือใช้ cron:  0 * * * *  cd /path/to/Consolidated_Weather && python3 collector.py run >> logs/collector.log 2>&1
```
**แบบ C — GitHub Actions (แนะนำที่สุด ไม่ต้องเปิดเครื่อง)**
GitHub รัน `collector.py` ให้ทุกชั่วโมง แล้วเก็บไว้ใน branch `data` ดูวิธีตั้งค่าใน README.md หัวข้อ "ดึงอัตโนมัติด้วย GitHub Actions"
ข้อมูลที่เก็บไว้จะถูกดึงลงเครื่องให้เองในข้อ ⑩ (หรือรัน `python3 collector.py sync`)

ทุกแบบบันทึกลงฐานข้อมูลเดียวกัน ยิ่งเก็บถี่และนานเท่าไร การวิเคราะห์ความแม่นยำที่ช่วง 3/6/12 ชม. ก็ยิ่งมีข้อมูลมาก

In [ ]:
AUTO = {"thread": None, "stop": threading.Event(), "last": None}
auto_out = W.Output(layout={"border": "1px solid #ccc", "padding": "4px"})
auto_status = W.HTML()


def _dashboard():
    # แดชบอร์ดย่อ: สรุป D+1 และฝนช่วงสั้นของทุกตำแหน่ง
    with auto_out:
        clear_output(wait=True)
        for lat, lon, name in LOCATIONS:
            r = core.analyze_location(lat, lon, name, CFG)
            RESULTS[name] = r
            store.save_snapshot(r, CFG)
            render_location(r, charts=False)


def _worker(every_min):
    while not AUTO["stop"].is_set():
        try:
            _dashboard()
            AUTO["last"] = pd.Timestamp.now()
            auto_status.value = f"✅ อัปเดตล่าสุด {AUTO['last']:%Y-%m-%d %H:%M:%S} · รอบถัดไปในอีก {every_min} นาที"
        except Exception as e:
            auto_status.value = f"⚠️ {pd.Timestamp.now():%H:%M} error: {e}"
        AUTO["stop"].wait(every_min * 60)


def start_auto_update(every_min=None):
    every_min = every_min or CFG["collect_every_minutes"]
    stop_auto_update()
    AUTO["stop"] = threading.Event()
    AUTO["thread"] = threading.Thread(target=_worker, args=(every_min,), daemon=True)
    AUTO["thread"].start()
    auto_status.value = f"⏳ เริ่มอัปเดตอัตโนมัติทุก {every_min} นาที…"


def stop_auto_update():
    if AUTO["thread"] and AUTO["thread"].is_alive():
        AUTO["stop"].set()
        auto_status.value = "⏹️ หยุดอัปเดตอัตโนมัติแล้ว"


every_box = W.BoundedIntText(value=CFG["collect_every_minutes"], min=10, max=1440, description="ทุก (นาที)")
btn_start = W.Button(description="▶️ เริ่มอัปเดตอัตโนมัติ", button_style="success", layout={"width": "200px"})
btn_stop = W.Button(description="⏹️ หยุด", layout={"width": "100px"})
btn_start.on_click(lambda _: start_auto_update(every_box.value))
btn_stop.on_click(lambda _: stop_auto_update())
display(W.VBox([W.HBox([every_box, btn_start, btn_stop]), auto_status, auto_out]))

In [ ]:
# ประวัติการดึงข้อมูลที่เก็บในฐานข้อมูล
runs = store.load_runs(CFG)
display(Markdown(f"เก็บแล้ว **{len(runs)}** รอบ · ล่าสุด {runs.fetched_at.max() if len(runs) else '–'}"))
runs.tail(10)

## ⑨ บันทึกว่าฝนตกจริงหรือไม่

เลือกตำแหน่งและเวลา (ค่าเริ่มต้นคือ "ตอนนี้") กด **ตก / ไม่ตก** ใส่ปริมาณฝนถ้ามีเครื่องวัด (ไม่ใส่ก็ได้) แล้วกด **บันทึก**

- บันทึกได้ทั้ง **เวลาปัจจุบัน** และ **ย้อนหลัง** (เช่น "เมื่อวาน 16:00 ฝนตกหนัก")
- **แนะนำให้บันทึกตอนที่ฝนไม่ตกด้วย** เพราะช่วยวัดว่าโมเดลไหน "ทายว่าตกแต่ไม่ตก" (false alarm) บ่อยแค่ไหน
- บันทึกจาก command line ก็ได้: `python3 collector.py observe --place me --rain yes --mm 5`

In [ ]:
_loc_options = [(f"{n} ({la:.2f}, {lo:.2f})", (la, lo, n)) for la, lo, n in LOCATIONS] + [("📌 กำหนดพิกัดเอง", None)]
now_ = pd.Timestamp.now()

w_loc = W.Dropdown(options=_loc_options, description="ตำแหน่ง", layout={"width": "420px"})
w_lat = W.FloatText(description="lat", layout={"width": "200px", "display": "none"})
w_lon = W.FloatText(description="lon", layout={"width": "200px", "display": "none"})
w_name = W.Text(description="ชื่อ", placeholder="เช่น ที่ทำงาน", layout={"width": "300px", "display": "none"})
w_date = W.DatePicker(description="วันที่", value=now_.date())
w_time = W.Text(description="เวลา", value=f"{now_:%H:%M}", placeholder="HH:MM", layout={"width": "180px"})
w_now = W.Button(description="🕒 ตอนนี้", layout={"width": "90px"})
w_rain = W.ToggleButtons(options=[("🌧️ ตก", True), ("☀️ ไม่ตก", False)], description="ฝน")
w_mm = W.Text(description="ปริมาณ (มม.)", placeholder="ไม่บังคับ", layout={"width": "220px"})
w_note = W.Text(description="หมายเหตุ", placeholder="เช่น ตกหนักมาก 20 นาที", layout={"width": "420px"})
w_save = W.Button(description="💾 บันทึก", button_style="primary")
w_msg = W.HTML()
obs_out = W.Output()
w_del_id = W.IntText(description="ลบ #", layout={"width": "150px"})
w_del = W.Button(description="🗑️ ลบ", button_style="danger", layout={"width": "80px"})


def _toggle_custom(change):
    show = "flex" if change["new"] is None else "none"
    for w in (w_lat, w_lon, w_name):
        w.layout.display = show


def _set_now(_):
    t = pd.Timestamp.now()
    w_date.value, w_time.value = t.date(), f"{t:%H:%M}"


def _show_obs():
    with obs_out:
        clear_output(wait=True)
        obs = store.load_observations(CFG)
        display(Markdown(f"**บันทึกทั้งหมด {len(obs)} รายการ** (ล่าสุด 15 รายการ)"))
        if len(obs):
            view = obs.assign(ฝน=obs.rained.map({1: "🌧️ ตก", 0: "☀️ ไม่ตก"}))
            display(view[["obs_id", "obs_time", "location", "ฝน", "amount_mm", "note"]].tail(15)
                    .set_index("obs_id"))


def _save(_):
    try:
        loc = w_loc.value or (w_lat.value, w_lon.value, w_name.value or f"{w_lat.value:.3f},{w_lon.value:.3f}")
        t = pd.Timestamp(f"{w_date.value} {w_time.value}")
        mm = float(w_mm.value) if w_mm.value.strip() else None
        oid = store.add_observation(loc[2], loc[0], loc[1], w_rain.value, mm, t, w_note.value, CFG)
        w_msg.value = f"✅ บันทึก #{oid}: {loc[2]} · {t:%Y-%m-%d %H:%M} · {'ฝนตก' if w_rain.value else 'ฝนไม่ตก'}"
        w_mm.value = w_note.value = ""
        _show_obs()
    except Exception as e:
        w_msg.value = f"⚠️ {e}"


def _delete(_):
    store.delete_observation(w_del_id.value, CFG)
    w_msg.value = f"🗑️ ลบ #{w_del_id.value} แล้ว"
    _show_obs()


w_loc.observe(_toggle_custom, names="value")
w_now.on_click(_set_now)
w_save.on_click(_save)
w_del.on_click(_delete)

display(W.VBox([
    w_loc, W.HBox([w_lat, w_lon, w_name]),
    W.HBox([w_date, w_time, w_now]),
    w_rain, W.HBox([w_mm]), w_note,
    W.HBox([w_save, w_msg]),
    obs_out,
    W.HBox([w_del_id, w_del]),
]))
_show_obs()

## ⑩ วิเคราะห์ย้อนกลับ: โมเดลไหนแม่น?

**วิธีตัดสิน** (รายละเอียดใน docs/MODELS.md หัวข้อ "การวัดความแม่นยำ")
- สำหรับการสังเกตแต่ละครั้ง ดูฝนที่แต่ละโมเดลพยากรณ์ในช่วง `verify_window_hours` ชม. รอบเวลานั้น
  ถ้ารวมกันได้ ≥ `verify_threshold_mm` ถือว่าโมเดล **"ทายว่าฝนตก"** ส่วน ensemble ถือว่าทายว่าตกเมื่อโอกาส ≥ 50%
- **แหล่งพยากรณ์ย้อนหลัง** มี 2 แหล่ง
  1. `collector` คือรอบที่เราเก็บไว้เอง ได้ครบทุกช่วงเวลาล่วงหน้า รวมถึง 3/6/12 ชม.
  2. `previous_runs` คือ Open-Meteo Previous Runs API ได้พยากรณ์ที่ออกล่วงหน้า 1–7 วัน **ย้อนหลังได้ทันที** ไม่ต้องรอเก็บเอง
- ในแต่ละช่วงเวลาล่วงหน้า ใช้เฉพาะรอบที่ใกล้เวลาจริงที่สุด

| คะแนน | ความหมาย | ดีเมื่อ |
|---|---|---|
| ทายถูก % | ทายว่าตกหรือไม่ตกได้ถูก | สูง |
| POD % | เมื่อฝนตกจริง โมเดลทายถูกกี่ % | สูง |
| FAR % | เมื่อโมเดลว่าตก แต่จริง ๆ ไม่ตกกี่ % | ต่ำ |
| CSI % | ทายว่าตกถูก เทียบกับทุกครั้งที่ตกจริงหรือทายว่าตก | สูง |
| Bias | >1 ทายว่าตกบ่อยเกินไป, <1 ทายว่าตกน้อยเกินไป | ≈ 1 |
| MAE มม. | คลาดเคลื่อนของปริมาณฝน (เฉพาะครั้งที่ใส่ปริมาณ) | ต่ำ |
| Brier | ความคลาดเคลื่อนของความน่าจะเป็น (ensemble) | ต่ำ (0 = สมบูรณ์) |

> ⚠️ ต้องมีการสังเกต **อย่างน้อย 20–30 ครั้ง** ทั้งที่ตกและไม่ตก ผลจึงเริ่มน่าเชื่อถือ ช่วงแรกให้ดูค่า `n` ประกอบเสมอ

In [ ]:
# ดึงข้อมูลที่ GitHub Actions เก็บไว้ (branch data) มารวมในฐานข้อมูลในเครื่องก่อน
import collector
try:
    collector.cmd_sync(None)
except Exception as e:
    print(f"ℹ️ ข้ามการ sync จาก GitHub: {e}")

In [ ]:
REC = store.verification_records(CFG, use_previous_runs=True)
if REC.empty:
    display(Markdown("ℹ️ ยังไม่มีข้อมูลให้เปรียบเทียบ ให้บันทึกฝนจริงในข้อ ⑨ ก่อน "
                     "(ถ้าบันทึกย้อนหลัง 1–7 วัน จะได้ผลจาก Previous Runs API ทันที)"))
else:
    DET_ALL, ENS_ALL = store.verification_scores(REC, by_lead=False)
    DET_LEAD, ENS_LEAD = store.verification_scores(REC, by_lead=True)
    display(Markdown(f"การสังเกต **{REC.obs_id.nunique()}** ครั้ง "
                     f"(ตก {int(REC.drop_duplicates('obs_id').observed.sum())} / "
                     f"ไม่ตก {int((~REC.drop_duplicates('obs_id').observed).sum())})"))
    display(Markdown("### 🏆 อันดับโมเดล deterministic (ทุกช่วงเวลาล่วงหน้ารวมกัน เรียงตาม CSI)"))
    display(style_tbl(DET_ALL[["n", "ทายถูก %", "POD %", "FAR %", "CSI %", "Bias", "MAE มม."]], "RdYlGn", "{:.1f}"))
    if not ENS_ALL.empty:
        display(Markdown("### 🎲 Ensemble (Brier ยิ่งต่ำยิ่งดี)"))
        display(ENS_ALL[["n", "ทายถูก %", "POD %", "FAR %", "Brier"]].sort_values("Brier").round(3))

In [ ]:
# Heatmap: ความแม่นยำแยกโมเดล × ช่วงเวลาล่วงหน้า
if not REC.empty:
    for metric, cmap in [("ทายถูก %", "RdYlGn"), ("CSI %", "RdYlGn")]:
        piv = DET_LEAD[metric].unstack("lead_bin").reindex(columns=[c for c in store.LEAD_LABELS
                                                                     if c in DET_LEAD.index.get_level_values(1)])
        n = DET_LEAD["n"].unstack("lead_bin").reindex_like(piv)
        plot_heatmap(piv.loc[DET_ALL.index], f"{metric} แยกโมเดล × ช่วงเวลาล่วงหน้า", unit="%",
                     cmap=cmap, vmax=100, counts=n.loc[DET_ALL.index])

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    DET_ALL[["ทายถูก %", "CSI %"]].plot.barh(ax=axes[0])
    axes[0].invert_yaxis()
    axes[0].set_title("อันดับโมเดล (ทุกช่วงเวลา)")
    axes[0].grid(axis="x", alpha=0.3)
    if not ENS_ALL.empty:
        ENS_ALL["Brier"].sort_values().plot.barh(ax=axes[1], color="seagreen")
        axes[1].invert_yaxis()
        axes[1].set_title("Ensemble Brier score (ต่ำ = ดี)")
        axes[1].grid(axis="x", alpha=0.3)
    else:
        axes[1].axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# รายละเอียดรายครั้ง: แต่ละโมเดลพยากรณ์ไว้เท่าไร (มม.) เทียบกับที่เกิดจริง
if not REC.empty:
    det = REC[REC.kind == "det"]
    detail = det.pivot_table(index=["obs_time", "location", "observed", "lead_bin"], columns="model",
                             values="value", observed=True)
    display(Markdown(f"**ฝนพยากรณ์ในช่วง {CFG['verify_window_hours']} ชม. รอบเวลาที่สังเกต (มม.)** "
                     f"— สีเขียว = ทายถูก, สีแดง = ทายผิด (threshold {CFG['verify_threshold_mm']} มม.)"))

    def _color(df):
        obs = df.index.get_level_values("observed").values[:, None]
        pred = df.values >= CFG["verify_threshold_mm"]
        ok = pred == obs
        return pd.DataFrame(np.where(np.isnan(df.values), "", np.where(ok, "background-color:#c7e9c0",
                                                                       "background-color:#fcbba1")),
                            index=df.index, columns=df.columns)
    try:
        display(detail.tail(60).style.apply(_color, axis=None).format("{:.1f}", na_rep="–"))
    except ImportError:
        display(detail.tail(60))
    REC.to_csv(OUTPUT_DIR / "verification_records.csv", index=False, encoding="utf-8-sig")

## 📝 ขั้นต่อไปสำหรับทำแอป

- `weather_core.py` และ `weather_store.py` ใช้ได้ทันทีกับ **Streamlit** (`st.dataframe`, `streamlit-folium`) หรือ **FastAPI** ถ้าทำเป็นแอปมือถือ ให้ใช้ปุ่ม "ตก/ไม่ตก" แบบเดียวกับข้อ ⑨ และใช้ GPS ของเครื่องแทน IP
- Open-Meteo ใช้ฟรีได้เฉพาะงาน **ไม่เชิงพาณิชย์** (~10,000 calls/วัน) ถ้าจะใช้เชิงพาณิชย์ต้องสมัครแผน API หรือ self-host (ดู docs/MODELS.md)
- ถ้าอยากได้ "ค่าจริง" อัตโนมัติโดยไม่ต้องกรอกเอง อาจใช้ข้อมูลสถานีของกรมอุตุนิยมวิทยา (TMD API) หรือฝนจากดาวเทียม GPM IMERG เป็นแหล่งเพิ่มเติม